In [1]:
import pandas as pd
import requests as requests

In [2]:
#url = "https://api-sdp.nwslsoccer.com/v1/nwsl/football/seasons/nwsl::Football_Season::0b6761e4701749f593690c0f338da74c/stats/players?locale=en-US&category=general&role=all&direction=desc&page=1&pageNumElement=400"

url = "https://api-sdp.nwslsoccer.com/v1/nwsl/football/seasons/nwsl::Football_Season::fad050beee834db88fa9f2eb28ce5a5c/stats/players?locale=en-US&category=general&role=all&direction=desc&page=1&pageNumElement=400"

session = requests.Session()
response = session.get(url)

data = response.json()

In [3]:
df = pd.json_normalize(data["players"])
df = df.explode("stats")
print(df.columns)

Index(['stats', 'rankLabel', 'playerId', 'providerId', 'bibNumber',
       'roleLabel', 'role', 'mediaFirstName', 'mediaLastName', 'shirtName',
       'shortName', 'displayName', 'nationality', 'nationalityIsoCode',
       'apiCallRequestTime', 'team.teamId', 'team.providerId',
       'team.shortName', 'team.officialName', 'team.acronymName',
       'team.acronymNameLocalized', 'team.isTeamFake', 'team.mediaName',
       'team.mediaShortName', 'team.countryCode', 'team.teamType',
       'team.overallSummary', 'team.stadium', 'team.allSeasonImagery',
       'team.editorial.social.facebook', 'team.editorial.social.instagram',
       'team.editorial.social.x', 'team.editorial.social.tikTok',
       'team.editorial.social.youTube', 'team.editorial.social.linkedIn',
       'team.editorial.websiteUrl', 'team.editorial.shopUrl',
       'team.editorial.ticketsUrl', 'team.editorial.clubPrimaryColour',
       'team.editorial.clubSecondaryColour', 'team.editorial.clubTextColour',
       'editoria

In [4]:
# Cleaning Data
# Turning Unstructured to Structured data

stats = pd.json_normalize(df["stats"])
df = pd.concat([df.drop(columns=["stats"]), stats], axis=1)
print(df.columns)

Index(['rankLabel', 'playerId', 'providerId', 'bibNumber', 'roleLabel', 'role',
       'mediaFirstName', 'mediaLastName', 'shirtName', 'shortName',
       'displayName', 'nationality', 'nationalityIsoCode',
       'apiCallRequestTime', 'team.teamId', 'team.providerId',
       'team.shortName', 'team.officialName', 'team.acronymName',
       'team.acronymNameLocalized', 'team.isTeamFake', 'team.mediaName',
       'team.mediaShortName', 'team.countryCode', 'team.teamType',
       'team.overallSummary', 'team.stadium', 'team.allSeasonImagery',
       'team.editorial.social.facebook', 'team.editorial.social.instagram',
       'team.editorial.social.x', 'team.editorial.social.tikTok',
       'team.editorial.social.youTube', 'team.editorial.social.linkedIn',
       'team.editorial.websiteUrl', 'team.editorial.shopUrl',
       'team.editorial.ticketsUrl', 'team.editorial.clubPrimaryColour',
       'team.editorial.clubSecondaryColour', 'team.editorial.clubTextColour',
       'editorial.playerR

In [5]:
print(df.head(5))

  rankLabel                                           playerId  \
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   

                              providerId bibNumber   roleLabel  role  \
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   

  mediaFirstName mediaLastName shirtName shortName  ...  \
0          Julie         Doyle            J. Doyle  ...   
0          Julie         Doyle            J. Doyle  

In [6]:
df_final = df.pivot_table(
    index="playerId",
    columns="statsLabel",
    values="statsValue"
).reset_index()
print(df_final.columns)

Index(['playerId', 'Accurate pass percentage', 'Aerial Duels',
       'Aerial Duels Won Percentage', 'Aerial Duels lost', 'Aerial Duels won',
       'Aerials Won Percentage', 'Appearances', 'Assists',
       'Assists (Intentional)',
       ...
       'Unsuccessful Passes Own Half', 'Unsuccessful Short Passes',
       'Unsuccessful lay-offs', 'Winning Goal', 'XGEfficiency', 'Xg',
       'Yellow Cards', 'Yellow Red Cards', 'Yellow cards', 'corners'],
      dtype='str', name='statsLabel', length=174)


In [7]:

print(df_final.head(5))

statsLabel                                           playerId  \
0           nwsl::Football_Player::0021a2896ee54ef191fb324...   
1           nwsl::Football_Player::01429efe66554be8bc7f86e...   
2           nwsl::Football_Player::021370e22b6846a48f5e988...   
3           nwsl::Football_Player::0441c191624b4049911e4bb...   
4           nwsl::Football_Player::04a1aad603eb46dd9c349c2...   

statsLabel  Accurate pass percentage  Aerial Duels  \
0                               61.0          31.0   
1                               75.0          27.0   
2                               84.0          27.0   
3                               69.0           6.0   
4                               68.0          36.0   

statsLabel  Aerial Duels Won Percentage  Aerial Duels lost  Aerial Duels won  \
0                                 54.84               14.0              17.0   
1                                 59.26               11.0              16.0   
2                                 77.78     

In [8]:
df = pd.merge(df.drop(columns=['statsId', 'statsLabel',
       'statsLabelAbbreviation', 'statsValue', 'statsUnit',
       'statsUnitAbbreviation']), df_final, on="playerId")
df = df.drop_duplicates(["playerId"]).reset_index(drop=True)
print(df.columns)

Index(['rankLabel', 'playerId', 'providerId', 'bibNumber', 'roleLabel', 'role',
       'mediaFirstName', 'mediaLastName', 'shirtName', 'shortName',
       ...
       'Unsuccessful Passes Own Half', 'Unsuccessful Short Passes',
       'Unsuccessful lay-offs', 'Winning Goal', 'XGEfficiency', 'Xg',
       'Yellow Cards', 'Yellow Red Cards', 'Yellow cards', 'corners'],
      dtype='str', length=214)


In [9]:
print(df.head(5))

  rankLabel                                           playerId  \
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
1      None  nwsl::Football_Player::a86e9ba2f4c44ce789c593b...   
2      None  nwsl::Football_Player::ff16e70c73b943e38662bc6...   
3      None  nwsl::Football_Player::0021a2896ee54ef191fb324...   
4      None  nwsl::Football_Player::021370e22b6846a48f5e988...   

                              providerId bibNumber   roleLabel  role  \
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
1  opta:Player:d9dirjpaqkq3gga2zyeykcv6d         1  Goalkeeper     1   
2  opta:Player:9yvkpwm78jaqr321imfc3rrbo         6    Defender     2   
3  opta:Player:e2obgph6szyo04em3qr94g1qt        22     Forward     4   
4  opta:Player:bilgoekg6gqwr3qbp6dp0j1zp         4    Defender     2   

   mediaFirstName   mediaLastName     shirtName     shortName  ...  \
0           Julie           Doyle                    J. Doyle  ...   
1          Aubrey       Kingsb

In [10]:
##QC

In [11]:
print(df.shape)

(400, 214)


In [12]:
df.loc[df["playerId"].isnull()==True]

,rankLabel,playerId,providerId,bibNumber,roleLabel,role,mediaFirstName,mediaLastName,shirtName,shortName,...,Unsuccessful Passes Own Half,Unsuccessful Short Passes,Unsuccessful lay-offs,Winning Goal,XGEfficiency,Xg,Yellow Cards,Yellow Red Cards,Yellow cards,corners


In [13]:
df.loc[df["shortName"].isnull()==True]

,rankLabel,playerId,providerId,bibNumber,roleLabel,role,mediaFirstName,mediaLastName,shirtName,shortName,...,Unsuccessful Passes Own Half,Unsuccessful Short Passes,Unsuccessful lay-offs,Winning Goal,XGEfficiency,Xg,Yellow Cards,Yellow Red Cards,Yellow cards,corners


In [14]:
df_test = df[[
    "playerId","roleLabel","shortName" ,"nationality","team.officialName","Appearances","Games Played","Minutes played","Interceptions","Ground Duels", "Ground Duels won","Duels","Duels won","Aerial Duels","Aerial Duels won","Blocked Shots","Blocks","Foul Attempted Tackle","Times Tackled","Last Player Tackle","Total Clearances","Total Tackles","Tackles won","Accurate pass percentage","Backward Passes","Final Third Touches","Forward Passes","Overruns","Progressive Carries","Recoveries","Second Goal Assists","Successful Launches","Successful Long Passes","Successful Passes Opposition Half","Unsuccessful Launches","Unsuccessful Long Passes","Unsuccessful Passes Opposition Half","Total Big Chances Created","Key Passes (Attempt Assists)","Assists (Intentional)","Attempts from Set Pieces","Away Goals","Goals","Home Goals","Corners Taken (incl short corners)","Corners Won","corners","Goal Assists","On target scoring attempts","Total Shots","Shots On Target ( inc goals )","Total Big Chances Scored","Total attacking assists","Total scoring attempts","XGEfficiency","Xg"]]

In [15]:
print(df_test.head(5))

                                            playerId   roleLabel  \
0  nwsl::Football_Player::4cb80ed654ff46e89167791...  Midfielder   
1  nwsl::Football_Player::a86e9ba2f4c44ce789c593b...  Goalkeeper   
2  nwsl::Football_Player::ff16e70c73b943e38662bc6...    Defender   
3  nwsl::Football_Player::0021a2896ee54ef191fb324...     Forward   
4  nwsl::Football_Player::021370e22b6846a48f5e988...    Defender   

      shortName nationality  team.officialName  Appearances  Games Played  \
0      J. Doyle         USA      Orlando Pride         13.0          13.0   
1  A. Kingsbury         USA  Washington Spirit         28.0          28.0   
2       E. Sams         USA      Orlando Pride         27.0          27.0   
3      B. Banda      Zambia      Orlando Pride         16.0          16.0   
4      Rafaelle      Brazil      Orlando Pride         14.0          14.0   

   Minutes played  Interceptions  Ground Duels  ...  corners  Goal Assists  \
0           528.0            1.0          46.0  ..

In [16]:
print(df_test["roleLabel"].unique())

<StringArray>
['Midfielder', 'Goalkeeper', 'Defender', 'Forward']
Length: 4, dtype: str


In [17]:
df_test = df_test.loc[df_test["roleLabel"]=='Defender']
print(df_test.head(5))

                                             playerId roleLabel  \
2   nwsl::Football_Player::ff16e70c73b943e38662bc6...  Defender   
4   nwsl::Football_Player::021370e22b6846a48f5e988...  Defender   
6   nwsl::Football_Player::521bad7c3a2f49a185c968f...  Defender   
9   nwsl::Football_Player::f0a281901a6340798be863c...  Defender   
15  nwsl::Football_Player::18d7e7320d2949b083c7315...  Defender   

        shortName nationality  team.officialName  Appearances  Games Played  \
2         E. Sams         USA      Orlando Pride         27.0          27.0   
4        Rafaelle      Brazil      Orlando Pride         14.0          14.0   
6   H. McCutcheon         USA      Orlando Pride         28.0          28.0   
9        K. Sylla      France  Washington Spirit         15.0          15.0   
15      E. Morgan     England  Washington Spirit         28.0          28.0   

    Minutes played  Interceptions  Ground Duels  ...  corners  Goal Assists  \
2           2362.0           36.0         1

In [18]:
#Movement

In [19]:
movement = df_test[[
    "playerId","roleLabel","shortName" ,"nationality","team.officialName","Appearances","Games Played","Minutes played","Interceptions","Duels","Duels won","Blocked Shots","Blocks","Times Tackled","Last Player Tackle","Total Clearances","Total Tackles","Tackles won","Foul Attempted Tackle"]]

per90_col = ["Interceptions","Duels","Duels won","Blocked Shots","Blocks","Foul Attempted Tackle","Times Tackled","Last Player Tackle","Total Clearances","Total Tackles","Tackles won"]
for col in per90_col:
    movement[col + " per90"] = movement[col]/movement["Minutes played"]*90

print(movement.head(5))

                                             playerId roleLabel  \
2   nwsl::Football_Player::ff16e70c73b943e38662bc6...  Defender   
4   nwsl::Football_Player::021370e22b6846a48f5e988...  Defender   
6   nwsl::Football_Player::521bad7c3a2f49a185c968f...  Defender   
9   nwsl::Football_Player::f0a281901a6340798be863c...  Defender   
15  nwsl::Football_Player::18d7e7320d2949b083c7315...  Defender   

        shortName nationality  team.officialName  Appearances  Games Played  \
2         E. Sams         USA      Orlando Pride         27.0          27.0   
4        Rafaelle      Brazil      Orlando Pride         14.0          14.0   
6   H. McCutcheon         USA      Orlando Pride         28.0          28.0   
9        K. Sylla      France  Washington Spirit         15.0          15.0   
15      E. Morgan     England  Washington Spirit         28.0          28.0   

    Minutes played  Interceptions  Ground Duels  ...  Aerial Duels per90  \
2           2362.0           36.0         115.

In [23]:
movement[["Times Tackled","Total Tackles","Tackles won"]]

,Times Tackled,Total Tackles,Tackles won
2,12.0,46.0,30.0
4,6.0,15.0,9.0
6,36.0,60.0,37.0
9,2.0,8.0,7.0
15,12.0,39.0,22.0
...,...,...,...
383,1.0,3.0,1.0
384,1.0,6.0,4.0
385,2.0,10.0,6.0
390,9.0,25.0,14.0
